# YOLO médical v2 – classes nettoyées + datasets en plus (Kaggle)
**Panneau de droite → Add Input**, ajoute :
1. ton dataset `unified`
2. ton modèle `best_unified1.pt` (crée un dataset avec, ex. `yolo-unified1`)
3. les datasets publics (Add Input → cherche le nom) :
   - `jphajp/ur5e-srube-nurse-surgical-instruments` (classes en espagnol : bisturi, pinzas)
   - `dilavado/labeled-surgical-tools` (scalpels)
   - tout autre dataset YOLO de seringues / compresses : fusionné automatiquement s'il a un `data.yaml`.

Accelerator **GPU T4 x2**, Internet **On**, puis **Run All**. Ne recharge pas la page pendant l'entraînement.

In [ ]:
!pip install -q -U ultralytics
import torch; print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN -> active le GPU')

## 1. Classes gardées et correspondances de noms

In [ ]:
# Retirées : hors sujet (chantier, personnes, véhicules, états de déchets)
DROP = {'bloody_objects', 'machinery', 'no_hardhat', 'no_mask', 'no_safety_vest', 'person',
        'radioactive_objects', 'safety_cone', 'vehicle', 'hardhat_worn', 'safety_vest_worn'}

# Noms trouvés dans d'autres datasets -> notre classe
ALIASES = {
    'bisturi': 'scalpel', 'pinzas': 'tweezer_metal',
    'syringe': 'syringe', 'syringes': 'syringe', 'seringue': 'syringe', 'injection': 'syringe',
    'scalpel': 'scalpel', 'scalpels': 'scalpel', 'scalpel handle': 'scalpel', 'scalpel blade': 'blade', 'blade': 'blade',
    'tweezers': 'tweezer_metal', 'tweezer': 'tweezer_metal', 'forceps': 'tweezer_metal',
    'needle': 'needle', 'needles': 'needle',
    'gauze': 'gauze', 'gauze pad': 'gauze', 'compress': 'gauze',
    'bandage': 'bandage', 'bandages': 'bandage', 'band aid': 'bandage',
    'gloves': 'medical_gloves', 'glove': 'medical_gloves', 'surgical gloves': 'medical_gloves',
    'mask': 'mask_item', 'surgical mask': 'mask_item', 'face mask': 'mask_item',
    'thermometer': 'electronic_thermometer', 'pill': 'pill', 'pills': 'pill', 'capsule': 'capsule',
    'cotton swab': 'cotton_swab', 'test tube': 'test_tube', 'pipette': 'single_channel_pipette',
}
def norm(n): return str(n).strip().lower().replace('-', ' ').replace('_', ' ')
ALIASES = {norm(k): v for k, v in ALIASES.items()}

## 2. Trouver les datasets YOLO dans /kaggle/input

In [ ]:
import glob, os, yaml, shutil, zipfile
from pathlib import Path
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    zipfile.ZipFile(z).extractall('/kaggle/working/zips/' + Path(z).stem)

yamls = [p for p in glob.glob('/kaggle/input/**/data.yaml', recursive=True) + glob.glob('/kaggle/working/zips/**/data.yaml', recursive=True)
         if 'classification' not in p and 'obb' not in p]
weights = sorted(glob.glob('/kaggle/input/**/best_unified1*.pt', recursive=True)) or sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
print('Datasets trouvés :')
for y in yamls: print('  ', y)
print('Poids de départ :', weights[:1])
assert yamls, 'Aucun data.yaml : ajoute tes datasets avec Add Input'
UNIFIED = next(y for y in yamls if 'unified' in y)
unified_names = yaml.safe_load(open(UNIFIED))['names']
unified_names = list(unified_names.values()) if isinstance(unified_names, dict) else unified_names
KEEP = [n for n in unified_names if n not in DROP]
NEW_ID = {n: i for i, n in enumerate(KEEP)}
print(len(KEEP), 'classes gardées')

## 3. Fusionner : unified filtré + datasets externes renommés

In [ ]:
OUT = Path('/kaggle/working/merged')
shutil.rmtree(OUT, ignore_errors=True)
for s in ('train', 'val', 'test'):
    (OUT / s / 'images').mkdir(parents=True)
    (OUT / s / 'labels').mkdir(parents=True)

def split_of(rel):
    parts = [p.lower() for p in Path(rel).parts]
    if any(p in ('val', 'valid', 'validation') for p in parts): return 'val'
    if 'test' in parts: return 'test'
    return 'train'

def labels_dir(img_dir):
    parts = list(img_dir.parts)
    i = len(parts) - 1 - parts[::-1].index('images')
    parts[i] = 'labels'
    return Path(*parts)

MAX_EXTRA = 2000  # images max par dataset externe (garde l'entraînement court)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
for yi, ypath in enumerate(yamls):
    names = yaml.safe_load(open(ypath)).get('names', [])
    names = list(names.values()) if isinstance(names, dict) else list(names)
    is_unified = ypath == UNIFIED
    mapping = {}
    for i, n in enumerate(names):
        target = n if is_unified else ALIASES.get(norm(n))
        if target in NEW_ID:
            mapping[i] = NEW_ID[target]
    print(ypath)
    print('   classes reprises :', len(mapping) if is_unified else {names[k]: KEEP[v] for k, v in mapping.items()})
    if not mapping:
        continue
    root = Path(ypath).parent
    kept = 0
    for img_dir in [d for d in root.rglob('images') if d.is_dir()]:
        lbl_dir = labels_dir(img_dir)
        split = split_of(img_dir.relative_to(root))
        for img in img_dir.iterdir():
            if img.suffix.lower() not in IMG_EXT:
                continue
            lbl = lbl_dir / (img.stem + '.txt')
            lines = []
            if lbl.exists():
                for line in lbl.read_text().splitlines():
                    p = line.split()
                    # boîtes seulement (5 valeurs) ; polygones / OBB ignorés
                    if len(p) == 5 and p[0].isdigit() and int(p[0]) in mapping:
                        lines.append(' '.join([str(mapping[int(p[0])])] + p[1:]))
            if not lines and not is_unified:
                continue
            if not is_unified and kept >= MAX_EXTRA:
                break
            stem = f'd{yi}_{img.stem}'
            shutil.copy(img, OUT / split / 'images' / (stem + img.suffix))
            (OUT / split / 'labels' / (stem + '.txt')).write_text('\n'.join(lines))
            kept += 1
    print(f'   {kept} images ajoutées')

yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
                'names': {i: n for i, n in enumerate(KEEP)}}, open('/kaggle/working/data.yaml', 'w'))
for s in ('train', 'val', 'test'):
    print(s, len(list((OUT / s / 'images').iterdir())), 'images')

## 3 bis. Labeled Surgical Tools (format non standard) : images scalpel seul

In [ ]:
import random
found = glob.glob('/kaggle/input/**/labeled-surgical-tools/Surgical-Dataset', recursive=True)
if found:
    root = Path(found[0])
    labels = {p.stem: p for p in (root / 'Labels').rglob('*.txt')}
    imgs = [p for p in (root / 'Images' / 'Scalpel').rglob('*')
            if p.suffix.lower() in IMG_EXT and not any(part.lower().startswith('with') for part in p.parts)]
    random.seed(0); kept = 0
    for img in imgs[:MAX_EXTRA]:
        lbl = labels.get(img.stem)
        if not lbl: continue
        boxes = [l.split() for l in lbl.read_text().splitlines()]
        lines = [' '.join([str(NEW_ID['scalpel'])] + b[1:]) for b in boxes if len(b) == 5]
        if not lines: continue
        split = 'val' if random.random() < 0.15 else 'train'
        shutil.copy(img, OUT / split / 'images' / ('lst_' + img.name))
        (OUT / split / 'labels' / ('lst_' + img.stem + '.txt')).write_text('\n'.join(lines))
        kept += 1
    print('Labeled Surgical Tools :', len(imgs), 'images scalpel seul,', kept, 'ajoutées')

## 4. Entraîner (reprend best_unified1, 50 epochs max, ~2 h 30)

In [ ]:
from ultralytics import YOLO
START = weights[0] if weights else 'yolo11s.pt'
print('Départ :', START)
model = YOLO(START)
model.train(
    data='/kaggle/working/data.yaml', epochs=50, imgsz=640, batch=32, patience=12,
    device=0 if torch.cuda.device_count() == 1 else [0, 1], workers=4,
    cos_lr=True, close_mosaic=10, scale=0.7, degrees=15, hsv_s=0.8, hsv_v=0.5, mixup=0.1, copy_paste=0.1,
    project='/kaggle/working/runs', name='unified2', exist_ok=True)

## 5. Score et export

In [ ]:
best = '/kaggle/working/runs/unified2/weights/best.pt'
m = YOLO(best).val(data='/kaggle/working/data.yaml')
print(f'mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}')
shutil.copy(best, '/kaggle/working/best_unified2.pt')
shutil.rmtree('/kaggle/working/merged', ignore_errors=True)
print('Télécharge best_unified2.pt dans Output (clic droit -> Download) AVANT de fermer.')